# LV Wall Thickness — AHA 17-Segment (ES only)

> **ED data is handled separately** (1,300 hearts already cached).
> This notebook focuses exclusively on **End-Systole** processing.

**Pipeline overview**
1. Load the UK Biobank Statistical Shape Model (SSM)
2. Collect real ES segmentations from ACDC / M&Ms / M&Ms-2
3. Fit the SSM to each patient's ES endo contours (L-BFGS-B)
4. Generate epi surfaces via normal offset
5. Build an ES occupancy cache
6. (Optional) Retrain the INR on the ES cache
7. **Compute AHA 17-segment wall thickness at ES** from endo/epi meshes

### Key fixes vs. original notebook
| Issue | Fix |
|---|---|
| `ssm_regularise = 0.01` too tight for ES | Drop to `0.001` |
| Only ACDC processed (M&Ms paths wrong) | Auto-detect paths, skip gracefully |
| No wall thickness code | Full AHA 17-seg WT + bull's eye plot added |
| **ACDC GT not found** (`*_gt.nii.gz` glob) | **`resolve_nii_path` + `_find_gt_entries`**: handles nested-directory `.nii` layout |

| `Info.cfg` missing → all patients skipped | Auto-detect ED/ES from LV voxel volume |

## 1 · Install Dependencies

In [ ]:
%pip install -q nibabel scikit-image trimesh scipy scikit-learn tqdm vtk matplotlib

## 2 · Imports

In [ ]:
import os, glob, json, warnings, configparser
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.cm as mcm
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

import nibabel as nib
from skimage.measure import find_contours
from scipy.ndimage import label as nd_label
from scipy.spatial import cKDTree
from scipy.optimize import minimize
import trimesh
import vtk

warnings.filterwarnings('ignore')
np.random.seed(42)
print('✅ Imports OK')


## 3 · Configuration

`ssm_regularise_es = 0.001` — allows the optimiser to produce
the smaller, thicker-walled ES shapes.

`n_modes = 100` — enough modes to span the contracted ES shape space.


In [ ]:
TRIAL_MODE = False   # True → 2 patients per dataset

CFG = dict(
    # ── Dataset paths (set to None to skip) ──────────────
    acdc_dir   = '/kaggle/input/datasets/andrefce/realmri/training',
    mnms_dir   = None,   # auto-detected below
    mnms2_dir  = None,   # auto-detected below

    # ── Output paths ──────────────────────────────────────
    es_cache_dir     = '/kaggle/working/es_occupancy_cache',

    # ── Must match training config exactly ────────────────
    pts_per_ring  = 40,
    n_sax_slices  = 10,

    # ── SSM ───────────────────────────────────────────────
    n_modes            = 100,    # modes used for fitting
    ssm_regularise_es  = 0.001,  # lower = more deformation allowed for ES
    ssm_max_iter       = 300,

    # ── Epi generation ────────────────────────────────────
    epi_base_offset = 10.0,
    epi_apex_offset =  5.0,
    epi_noise_std   =  0.5,

    # ── Occupancy sampling ────────────────────────────────
    n_query     = 2048,
    surface_std = 2.0,

    # ── Segmentation labels (ACDC / M&Ms / M&Ms-2) ────────
    lbl_lv  = 3,
    lbl_myo = 2,
    lbl_rv  = 1,
    lbl_bg  = 0,

    seed = 42,
)

if TRIAL_MODE:
    CFG.update(n_query=512, ssm_max_iter=80)
    print('🧪 TRIAL MODE — 2 patients/dataset, 512 query pts, 80 SSM iters')
else:
    print('🚀 FULL MODE')

# ── Auto-detect M&Ms paths ────────────────────────────────────
_mnms_candidates = [
    '/kaggle/input/m-and-ms-dataset/MnM',
    '/kaggle/input/mnms/MnM',
    '/kaggle/input/mms-dataset/MnM',
    '/kaggle/input/m-ms-challenge/MnM',
]
_mnms2_candidates = [
    '/kaggle/input/datasets/andrefce/realdata2/MnM2/dataset',
    '/kaggle/input/m-and-m2-dataset/MnM2/dataset',
    '/kaggle/input/mnms2/MnM2/dataset',
    '/kaggle/input/mms2-dataset/dataset',
]
for p in _mnms_candidates:
    if os.path.isdir(p):
        CFG['mnms_dir'] = p; break
for p in _mnms2_candidates:
    if os.path.isdir(p):
        CFG['mnms2_dir'] = p; break

os.makedirs(CFG['es_cache_dir'], exist_ok=True)

for name, key in [('ACDC', 'acdc_dir'), ('M&Ms', 'mnms_dir'), ('M&Ms-2', 'mnms2_dir')]:
    p = CFG.get(key)
    if p and os.path.isdir(p):
        print(f'  ✅ {name}: {p}')
    else:
        print(f'  ⚠  {name}: not found — will be skipped')
        CFG[key] = None

## 4 · Load UK Biobank SSM

Clones the UK-Digital-Heart-Project repo and loads:
- **mean mesh** — defines the clean SSM topology used for all ED *and* ES samples
- **PCA modes** — displacement fields that deform the mean mesh

Four layout strategies are tried in order:
1. Separate `mode_NNN.vtk` files per mode
2. Point-data arrays embedded in the mean VTK file
3. `.npz` numpy archive alongside the VTK
4. **CSV/CSV.GZ** — the actual format shipped by the UK-Digital-Heart-Project repo
   (`LV_ED_pc_100_modes.csv.gz` + `LV_ED_var_100_modes.csv.gz`)

In [ ]:
SSM_DIR = 'Statistical-Shape-Model'
if not os.path.exists(SSM_DIR):
    ret = os.system('git clone --quiet '
                    'https://github.com/UK-Digital-Heart-Project/'
                    'Statistical-Shape-Model.git')
    print('✅ SSM cloned' if ret == 0 else '❌ git clone failed')

# ─── VTK mesh loader ─────────────────────────────────────────
def load_vtk_mesh(path):
    """Load VTK polydata, triangulate, return (vertices np.float64, faces np.int32)."""
    reader = vtk.vtkPolyDataReader()
    reader.SetFileName(str(path))
    reader.Update()
    poly = reader.GetOutput()
    tri = vtk.vtkTriangleFilter()
    tri.SetInputData(poly); tri.Update()
    poly = tri.GetOutput()
    pts = poly.GetPoints()
    verts = np.array([pts.GetPoint(i) for i in range(pts.GetNumberOfPoints())], dtype=np.float64)
    cells_ = poly.GetPolys(); cells_.InitTraversal()
    id_list = vtk.vtkIdList(); faces = []
    while cells_.GetNextCell(id_list):
        if id_list.GetNumberOfIds() == 3:
            faces.append([id_list.GetId(j) for j in range(3)])
    return verts, np.array(faces, dtype=np.int32)

# ─── SSM loader ──────────────────────────────────────────────
def load_ssm(ssm_dir, n_modes=100):
    """
    Returns
    -------
    mean_verts : (N, 3)
    faces      : (F, 3)  int32
    modes      : (M, N, 3)  displacement fields
    stds       : (M,)  sqrt(eigenvalue) — used for regularisation bounds
    """
    ssm_dir = Path(ssm_dir)

    # Find mean mesh (prefer LV endo)
    all_vtk = list(ssm_dir.rglob('*.vtk'))
    mean_path = None
    for c in all_vtk:
        nm = c.name.lower()
        if 'mean' in nm and ('lv' in nm or 'endo' in nm):
            mean_path = c; break
    if mean_path is None:
        for c in all_vtk:
            if 'mean' in c.name.lower():
                mean_path = c; break
    if mean_path is None and all_vtk:
        mean_path = all_vtk[0]
    if mean_path is None:
        raise FileNotFoundError(f'No VTK file found under {ssm_dir}')

    mean_verts, faces = load_vtk_mesh(mean_path)
    N = len(mean_verts)
    print(f'Mean mesh: {mean_path.name}  |  {N} verts, {len(faces)} faces')

    # ── Layout A: separate mode VTK files ────────────────────
    mode_files = sorted(
        list(ssm_dir.rglob('*mode_*.vtk')) +
        list(ssm_dir.rglob('*Mode_*.vtk')) +
        list(ssm_dir.rglob('*pc_*.vtk'))
    )[:n_modes]

    if mode_files:
        raw_modes, stds = [], []
        for mf in mode_files:
            mv, _ = load_vtk_mesh(mf)
            raw_modes.append(mv - mean_verts)
            stds.append(1.0)
        modes = np.stack(raw_modes)
        stds  = np.array(stds)
        print(f'Loaded {len(modes)} modes from separate VTK files')
        return mean_verts, faces, modes, stds

    # ── Layout B: point-data arrays in the mean VTK ──────────
    reader = vtk.vtkPolyDataReader()
    reader.SetFileName(str(mean_path)); reader.Update()
    pd = reader.GetOutput().GetPointData()
    raw_modes = []
    for i in range(pd.GetNumberOfArrays()):
        arr = pd.GetArray(i)
        if arr and arr.GetNumberOfComponents() == 3 and arr.GetNumberOfTuples() == N:
            raw_modes.append(np.array([arr.GetTuple3(j) for j in range(N)]))
        if len(raw_modes) >= n_modes:
            break
    if raw_modes:
        modes = np.stack(raw_modes)
        stds  = np.ones(len(raw_modes))
        print(f'Loaded {len(modes)} modes from VTK point-data arrays')
        return mean_verts, faces, modes, stds

    # ── Layout C: .npz archive ───────────────────────────────
    npzs = list(ssm_dir.rglob('*.npz'))
    if npzs:
        data  = np.load(npzs[0])
        modes = data.get('modes', data.get('components'))[:n_modes]
        stds  = data.get('stds',  data.get('singular_values',
                                            np.ones(len(modes))))[:n_modes]
        if modes.ndim == 2:          # (M, 3N) → (M, N, 3)
            modes = modes.reshape(len(modes), N, 3)
        print(f'Loaded {len(modes)} modes from {npzs[0].name}')
        return mean_verts, faces, modes, stds

    # ── Layout D: CSV / CSV.GZ (UK-Digital-Heart-Project format) ──
    # The repo ships:
    #   LV_ED_pc_100_modes.csv.gz  — shape (3N, M)  principal components
    #   LV_ED_var_100_modes.csv.gz — shape (M,)      eigenvalues
    csv_pc_files = sorted(
        list(ssm_dir.rglob('*_pc_*modes*.csv.gz')) +
        list(ssm_dir.rglob('*_pc_*modes*.csv'))
    )
    csv_var_files = sorted(
        list(ssm_dir.rglob('*_var_*modes*.csv.gz')) +
        list(ssm_dir.rglob('*_var_*modes*.csv'))
    )
    # Prefer LV files
    pc_file = None
    for cf in csv_pc_files:
        if 'lv' in cf.name.lower():
            pc_file = cf; break
    if pc_file is None and csv_pc_files:
        pc_file = csv_pc_files[0]

    var_file = None
    for vf in csv_var_files:
        if 'lv' in vf.name.lower():
            var_file = vf; break
    if var_file is None and csv_var_files:
        var_file = csv_var_files[0]

    if pc_file is not None:
        print(f'Loading PCs from {pc_file.name} …')
        pc = np.genfromtxt(str(pc_file), delimiter=',')   # (3N, M)
        M = min(n_modes, pc.shape[1])
        # Each column is one mode — reshape from (3N,) to (N, 3)
        modes = np.stack([pc[:, i].reshape(N, 3) for i in range(M)])   # (M, N, 3)

        if var_file is not None:
            variance = np.genfromtxt(str(var_file), delimiter=',')
            stds = np.sqrt(variance[:M])
            print(f'Loaded variances from {var_file.name}')
        else:
            stds = np.ones(M)
            print('⚠  No variance file found — using unit stds')

        print(f'Loaded {M} modes from CSV  (pc shape={pc.shape})')
        return mean_verts, faces, modes, stds

    # ── Fallback: warn, return zero modes ────────────────────
    # Diagnose what files *do* exist
    all_files = list(ssm_dir.rglob('*'))
    exts = defaultdict(int)
    for f in all_files:
        exts[f.suffix] += 1
    print(f'⚠  No PCA modes found under {ssm_dir}')
    print(f'   File types present: {dict(exts)}')
    modes = np.zeros((1, N, 3)); stds = np.ones(1)
    return mean_verts, faces, modes, stds

SSM_MEAN_VERTS, SSM_FACES, SSM_MODES, SSM_STDS = load_ssm(SSM_DIR, n_modes=CFG['n_modes'])
print(f'SSM loaded: {len(SSM_MODES)} modes, {len(SSM_MEAN_VERTS)} vertices')

## 5 · Mesh Geometry Helpers

In [ ]:
def compute_vertex_normals(verts, faces, orient_outward=True):
    """Per-vertex normals (area-weighted).

    When ``orient_outward=True`` (default) the normal field is flipped
    if it predominantly points inward — i.e. ``mean((v - v̄) · n) < 0``.
    This is essential for SSM-derived meshes: the template winding can
    be inward, which would otherwise cause ``generate_epi`` to push the
    epicardium *into* the cavity (the root cause of the every-sample
    label-swap reported by the v2 training notebook).
    """
    v0, v1, v2 = verts[faces[:,0]], verts[faces[:,1]], verts[faces[:,2]]
    fn = np.cross(v1 - v0, v2 - v0)        # area-weighted face normals
    vn = np.zeros_like(verts)
    for i in range(3):
        np.add.at(vn, faces[:, i], fn)
    norms = np.linalg.norm(vn, axis=1, keepdims=True)
    vn = vn / (norms + 1e-8)
    if orient_outward:
        radial = verts - verts.mean(axis=0, keepdims=True)
        if np.einsum("ij,ij->", radial, vn) < 0:
            vn = -vn
    return vn


def find_boundary_verts(faces):
    """Return indices of boundary (open rim) vertices."""
    edge_count = defaultdict(int)
    for f in faces:
        for i in range(3):
            e = tuple(sorted([f[i], f[(i+1) % 3]]))
            edge_count[e] += 1
    bv = set()
    for (a, b), cnt in edge_count.items():
        if cnt == 1:
            bv.add(a); bv.add(b)
    return np.array(sorted(bv), dtype=np.int32)


def find_apex_base(verts, faces):
    """Returns (apex_idx, base_vert_indices, base_centroid)."""
    bv          = find_boundary_verts(faces)
    base_center = verts[bv].mean(axis=0)
    dists       = np.linalg.norm(verts - base_center, axis=1)
    apex_idx    = int(np.argmax(dists))
    return apex_idx, bv, base_center


def pts_per_ring_resample(ring_pts, n=40):
    """Resample a closed contour ring to n evenly-spaced points."""
    ring = np.vstack([ring_pts, ring_pts[0]])
    diffs = np.diff(ring, axis=0)
    seg_lens = np.linalg.norm(diffs, axis=1)
    cum = np.concatenate([[0], np.cumsum(seg_lens)])
    total = cum[-1]
    if total < 1e-6:
        return np.tile(ring_pts[0], (n, 1))
    t_new = np.linspace(0, total, n, endpoint=False)
    out = np.zeros((n, ring_pts.shape[1]))
    for i, t in enumerate(t_new):
        idx = np.searchsorted(cum, t, side="right") - 1
        idx = min(idx, len(seg_lens) - 1)
        frac = (t - cum[idx]) / (seg_lens[idx] + 1e-8)
        out[i] = ring[idx] + frac * diffs[idx]
    return out


## 6 · Contour Extraction from NIfTI Segmentations

Extracts LV endo contour points (world-space mm) from a 3-D segmentation
volume for a given frame index.  Works for ACDC, M&Ms, and M&Ms-2 label
conventions (LV cavity = label 3).

**`resolve_nii_path`** — handles the nested-directory NIfTI layout used in
this dataset, where `patient001_frame01_gt.nii` is actually a *folder*
containing the real `.nii` file inside it.


In [ ]:
def resolve_nii_path(path):
    """Resolve the nested-folder NIfTI layout used in this dataset."""
    p = Path(path)
    if p.is_file():
        return p
    if p.is_dir():
        for pat in ("*.nii.gz", "*.nii"):
            hits = sorted(p.glob(pat))
            if hits:
                return hits[0]
        raise FileNotFoundError(f"No NIfTI file found inside directory: {p}")
    raise FileNotFoundError(f"Path does not exist: {p}")


def _extract_ring_stack(mask_3d, affine, n_pts=40, n_slices=10):
    """Extract one outer contour ring per axial slice from a binary mask.

    Returns an (P, 3) array of world-space mm coordinates, or ``None`` if
    the mask is empty or no rings could be drawn.
    """
    rings = []
    for z in range(mask_3d.shape[2]):
        sl = mask_3d[:, :, z]
        if sl.sum() < 10:
            continue
        try:
            contours = find_contours(sl.astype(float), 0.5)
        except Exception:
            continue
        if not contours:
            continue
        # Outer (largest) ring per slice
        c = max(contours, key=lambda x: len(x))
        n = len(c)
        vox = np.column_stack([c[:, 0], c[:, 1], np.full(n, z), np.ones(n)])
        world = (affine @ vox.T).T[:, :3]
        rings.append(world)

    if not rings:
        return None

    if len(rings) > n_slices:
        idxs = np.linspace(0, len(rings) - 1, n_slices, dtype=int)
        rings = [rings[i] for i in idxs]

    return np.vstack([pts_per_ring_resample(r, n_pts) for r in rings])


def extract_endo_contours(seg_vol, affine, lbl_lv=3, n_pts=40, n_slices=10):
    """LV cavity (endocardial) contour points — world-space mm.

    Uses ``lbl_lv`` only.  Returns ``None`` if the cavity is empty.
    """
    lv_mask = (seg_vol == lbl_lv).astype(np.uint8)
    if lv_mask.sum() == 0:
        return None
    return _extract_ring_stack(lv_mask, affine, n_pts, n_slices)


def extract_epi_contours(seg_vol, affine,
                          lbl_lv=3, lbl_myo=2,
                          n_pts=40, n_slices=10):
    """Epicardial contour points from the OUTER boundary of (LV ∪ myocardium).

    The epicardium is the outer surface of the myocardial wall.  In ACDC /
    M&Ms / M&Ms-2 segmentations this is the outer contour of the
    ``(seg == lbl_lv) | (seg == lbl_myo)`` mask.

    Returns ``None`` if myocardium is missing in the segmentation, in which
    case the caller must fall back to the (now correctly oriented)
    normal-offset epi from ``generate_epi``.
    """
    myo_mask = (seg_vol == lbl_myo)
    if myo_mask.sum() < 50:
        # Either the dataset lacks a myo label, or it's too sparse to trust.
        return None
    combined = (myo_mask | (seg_vol == lbl_lv)).astype(np.uint8)
    return _extract_ring_stack(combined, affine, n_pts, n_slices)


def extract_lv_contours(seg_vol, affine,
                         lbl_lv=3, lbl_myo=2,
                         n_pts=40, n_slices=10):
    """Both endo and (when myo label is present) real epi contours.

    Returns ``(endo_pts, epi_pts_or_None)``.
    """
    endo = extract_endo_contours(seg_vol, affine, lbl_lv, n_pts, n_slices)
    epi  = extract_epi_contours (seg_vol, affine, lbl_lv, lbl_myo,
                                  n_pts, n_slices)
    return endo, epi


def load_nifti_frame(nii_path, frame=0):
    """Load a single frame from a 3D or 4D NIfTI file/directory."""
    real_path = resolve_nii_path(nii_path)
    img  = nib.load(str(real_path))
    data = img.get_fdata(dtype=np.float32)
    if data.ndim == 4:
        data = data[..., frame]
    return data.astype(np.int32), img.affine


## 7 · SSM Fitting to ES Contours

Minimises the sum of squared distances from SSM vertices to the
nearest observed contour point plus a Mahalanobis-style regulariser
that keeps coefficients within ±3σ:

```
loss = mean_{v}(dist(v, contours)²) + λ · Σ_i (c_i / σ_i)²
```

`λ = 0.001` for ES (was 0.01 in the original notebook — the key fix).


In [ ]:
import torch, gc
from contextlib import nullcontext

NUM_GPUS = torch.cuda.device_count()
print(f"GPUs available: {NUM_GPUS}")
for i in range(NUM_GPUS):
    name = torch.cuda.get_device_name(i)
    mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name}  ({mem:.1f} GB)")

# ── Per-device SSM tensor cache ──────────────────────────────
# Uploaded ONCE per GPU; every patient reuses them (read-only).
_SSM_GPU_CACHE = {}

def _get_ssm_cached(device, mean_verts, modes, stds, n_modes):
    """Upload SSM arrays to a GPU once, then reuse for every patient."""
    key = str(device)
    if key not in _SSM_GPU_CACHE:
        M   = min(n_modes, len(modes))
        N   = len(mean_verts)
        dev = torch.device(device)
        _SSM_GPU_CACHE[key] = dict(
            mean_t  = torch.tensor(mean_verts.flatten(),      dtype=torch.float32, device=dev),
            modes_t = torch.tensor(modes[:M].reshape(M, -1),  dtype=torch.float32, device=dev),
            stds_t  = torch.tensor(stds[:M],                  dtype=torch.float32, device=dev),
            bounds_t= 3.5 * torch.tensor(stds[:M],            dtype=torch.float32, device=dev),
            M=M, N=N,
        )
        mb = _SSM_GPU_CACHE[key]['modes_t'].nbytes / 1e6
        print(f"  [SSM→{device}] cached {M} modes, {N} verts ({mb:.0f} MB)")
    return _SSM_GPU_CACHE[key]


def fit_ssm_to_contours(contour_pts, mean_verts, modes, stds,
                         n_modes=None, regularise=0.001, max_iter=300,
                         device=None):
    """
    GPU-accelerated SSM fitting via torch.optim.LBFGS.

    SSM tensors (mean, modes, stds) are cached per-GPU so that
    multiple workers share the same read-only memory.  Only the
    tiny per-patient target_t and coeffs are allocated fresh.
    """
    if device is None:
        device = "cuda:0" if torch.cuda.is_available() else "cpu"
    dev = torch.device(device)

    M_req = n_modes or len(modes)
    cache = _get_ssm_cached(device, mean_verts, modes, stds, M_req)
    mean_t   = cache['mean_t']
    modes_t  = cache['modes_t']
    stds_t   = cache['stds_t']
    bounds_t = cache['bounds_t']
    M, N     = cache['M'], cache['N']

    # ── Per-patient tensors (small) ──────────────────────────
    target_t = torch.tensor(contour_pts, dtype=torch.float32, device=dev)
    coeffs   = torch.zeros(M, device=dev, requires_grad=True)

    optimizer = torch.optim.LBFGS(
        [coeffs],
        max_iter=max_iter,
        history_size=20,
        tolerance_grad=1e-7,
        tolerance_change=1e-10,
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad()
        c     = torch.clamp(coeffs, -bounds_t, bounds_t)
        verts = (mean_t + modes_t.T @ c).reshape(N, 3)
        dists = torch.cdist(verts, target_t).min(dim=1).values
        loss  = dists.pow(2).mean() + regularise * (c / (stds_t + 1e-8)).pow(2).sum()
        loss.backward()
        return loss

    optimizer.step(closure)

    with torch.no_grad():
        c_final  = torch.clamp(coeffs, -bounds_t, bounds_t)
        fitted   = (mean_t + modes_t.T @ c_final).reshape(N, 3).cpu().numpy().astype(np.float64)
        c_np     = c_final.cpu().numpy()
        verts_   = (mean_t + modes_t.T @ c_final).reshape(N, 3)
        d_       = torch.cdist(verts_, target_t).min(dim=1).values
        loss_val = float(d_.pow(2).mean() + regularise * (c_final / (stds_t + 1e-8)).pow(2).sum())

    # ── Free per-patient GPU memory immediately ──────────────
    del target_t, coeffs, optimizer, c_final, verts_, d_

    return fitted, c_np, loss_val

## 8 · Epi Generation and Occupancy Sampling

In [ ]:
def generate_epi(endo_verts, faces,
                 base_offset=10.0, apex_offset=5.0, noise_std=0.5, seed=42):
    """Generate epi by offsetting endo along **outward**-oriented normals.

    The caller no longer needs to re-orient: ``compute_vertex_normals``
    flips the field if it points inward.  As a defensive post-condition
    we verify ``mean_radius(epi) > mean_radius(endo)``; if it fails (rare
    pathological mesh), we flip the normals and retry once.
    """
    rng     = np.random.default_rng(seed)
    normals = compute_vertex_normals(endo_verts, faces, orient_outward=True)

    _, _, base_center = find_apex_base(endo_verts, faces)
    apex_v   = endo_verts[np.argmax(np.linalg.norm(endo_verts - base_center, axis=1))]
    long_vec = base_center - apex_v
    long_len = np.linalg.norm(long_vec) + 1e-8
    t = np.clip(np.dot(endo_verts - apex_v, long_vec / long_len) / long_len, 0, 1)

    offset  = apex_offset + t * (base_offset - apex_offset)
    offset += rng.normal(0, noise_std, size=len(endo_verts))
    offset  = np.clip(offset, 2.0, 20.0)

    def _build(n):
        return endo_verts + n * offset[:, None]

    epi = _build(normals)

    # Post-condition: epi should sit OUTSIDE endo.
    cm     = 0.5 * (endo_verts.mean(0) + epi.mean(0))
    r_endo = np.linalg.norm(endo_verts - cm, axis=1).mean()
    r_epi  = np.linalg.norm(epi - cm,        axis=1).mean()
    if r_epi <= r_endo * 1.02:
        # Defensive flip — should be rare given orient_outward.
        epi = _build(-normals)

    return epi


def sample_occupancy(endo_verts, epi_verts, faces,
                     n_query=2048, surface_std=2.0, seed=42):
    """Sample query points and **two** binary occupancy masks.

    Schema matches ``build-lv-cache``:
      • ``endo_occ``  — 1 inside the endocardium (cavity)
      • ``epi_occ``   — 1 inside the epicardium  (cavity ∪ myocardial wall)
                        with ``epi_occ ≥ endo_occ`` enforced.

    Sampling distribution mirrors ED: 40 % near endo, 40 % near epi,
    20 % uniform inside the mesh bounding box.

    Returns
    -------
    pts      : (n_query, 3) float32  — WORLD coords (callers normalise)
    endo_occ : (n_query,)   float32  — ∈ {0, 1}
    epi_occ  : (n_query,)   float32  — ∈ {0, 1}
    """
    rng = np.random.default_rng(seed)

    n_surf = int(n_query * 0.40)
    n_rand = n_query - 2 * n_surf

    endo_idx = rng.integers(0, len(endo_verts), n_surf)
    epi_idx  = rng.integers(0, len(epi_verts),  n_surf)
    e_pts = (endo_verts[endo_idx]
             + rng.normal(0, surface_std, (n_surf, 3))).astype(np.float32)
    p_pts = (epi_verts[epi_idx]
             + rng.normal(0, surface_std, (n_surf, 3))).astype(np.float32)

    all_v  = np.vstack([endo_verts, epi_verts])
    lo, hi = all_v.min(0) - 5.0, all_v.max(0) + 5.0
    r_pts  = rng.uniform(lo, hi, (n_rand, 3)).astype(np.float32)

    pts = np.vstack([e_pts, p_pts, r_pts]).astype(np.float32)

    # Signed-distance via KD-tree + outward-oriented normals.
    endo_normals = compute_vertex_normals(endo_verts, faces, orient_outward=True)
    epi_normals  = compute_vertex_normals(epi_verts,  faces, orient_outward=True)

    endo_tree = cKDTree(endo_verts)
    epi_tree  = cKDTree(epi_verts)
    _, i_endo = endo_tree.query(pts)
    _, i_epi  = epi_tree.query(pts)

    sign_endo = np.einsum("ij,ij->i", pts - endo_verts[i_endo], endo_normals[i_endo])
    sign_epi  = np.einsum("ij,ij->i", pts - epi_verts[i_epi],   epi_normals[i_epi])

    inside_endo = (sign_endo < 0)
    inside_epi  = (sign_epi  < 0) | inside_endo   # anatomical: epi ⊇ endo

    return pts, inside_endo.astype(np.float32), inside_epi.astype(np.float32)


## 9 · Dataset Scanners

In [ ]:
# ──────────────────────────────────────────────────────────────
# ACDC scanner — handles BOTH layouts:
#   flat   : patient001/patient001_frame01_gt.nii.gz
#   nested : patient001/patient001_frame01_gt.nii/SomeFile.nii
# ──────────────────────────────────────────────────────────────
def _find_gt_entries(pat_dir):
    """
    Return a dict  frame_number → path  for all GT entries
    (file or directory) inside pat_dir.

    Supported name patterns:
      *_frameNN_gt.nii.gz
      *_frameNN_gt.nii        (flat file)
      *_frameNN_gt.nii/       (nested directory — this dataset's layout)
    """
    import re
    pat_dir = Path(pat_dir)
    entries = {}

    for child in sorted(pat_dir.iterdir()):
        m = re.search(r'_frame(\d+)_gt', child.name)
        if m is None:
            continue
        frame_no = int(m.group(1))
        # Accept both files and directories whose name matches *_gt.nii[.gz]
        if child.suffix in ('.gz', '.nii') or child.is_dir():
            entries[frame_no] = child

    return entries


def scan_acdc(acdc_dir):
    """
    Yield dicts with keys: patient_id, ed_gt, es_gt
    (paths to ground-truth NIfTI files/dirs for ED and ES frames).

    Supports:
      • Standard ACDC with Info.cfg and flat .nii.gz files
      • This dataset's nested layout (patient001_frame01_gt.nii/ directories)
      • Missing Info.cfg — auto-detects ED (max LV vol) and ES (min LV vol)
    """
    acdc_dir = Path(acdc_dir)
    for pat_dir in sorted(acdc_dir.glob('patient*')):
        if not pat_dir.is_dir():
            continue

        gt_entries = _find_gt_entries(pat_dir)
        if not gt_entries:
            continue          # no GT files in this folder

        # ── Try to read ED / ES frame numbers from Info.cfg ──
        ed_frame = es_frame = None
        cfg_path = pat_dir / 'Info.cfg'
        if cfg_path.exists():
            cfg = configparser.ConfigParser()
            try:
                with open(cfg_path) as _f:
                    cfg.read_string('[DEFAULT]\n' + _f.read())
                ed_frame = int(cfg['DEFAULT']['ED'])
                es_frame = int(cfg['DEFAULT']['ES'])
            except Exception:
                pass

        # ── Fallback: detect ED / ES from LV volume ──────────
        if ed_frame is None or es_frame is None:
            lv_vols = {}
            for fn, gt_path in gt_entries.items():
                try:
                    real = resolve_nii_path(gt_path)
                    img  = nib.load(str(real))
                    vol  = img.get_fdata(dtype=np.float32)
                    if vol.ndim == 4:
                        vol = vol[..., 0]
                    lv_vols[fn] = int((vol == CFG["lbl_lv"]).sum())
                except Exception:
                    lv_vols[fn] = 0

            if len(lv_vols) < 2:
                continue      # need at least 2 frames

            ed_frame = max(lv_vols, key=lv_vols.get)
            es_frame = min((k for k, v in lv_vols.items() if v > 0),
                           key=lv_vols.get, default=None)
            if es_frame is None or es_frame == ed_frame:
                continue

        if ed_frame not in gt_entries or es_frame not in gt_entries:
            continue

        yield dict(
            patient_id = pat_dir.name,
            ed_gt      = gt_entries[ed_frame],
            es_gt      = gt_entries[es_frame],
            source     = 'acdc',
        )


# ──────────────────────────────────────────────────────────────
# M&Ms scanner (4-D GT volumes)
# ──────────────────────────────────────────────────────────────
def _find_mnms_ed_es_frames(gt_4d):
    """
    In M&Ms the GT is 4D (X, Y, Z, T).
    ED = frame with largest LV volume; ES = smallest.
    """
    lv_vols = [(gt_4d[..., t] == 3).sum() for t in range(gt_4d.shape[3])]
    ed_frame = int(np.argmax(lv_vols))
    es_frame = int(np.argmin([v if v > 0 else 9e9 for v in lv_vols]))
    return ed_frame, es_frame


def scan_mnms(mnms_dir):
    mnms_dir = Path(mnms_dir)
    for gt_path in sorted(mnms_dir.rglob('*_gt.nii.gz')):
        img   = nib.load(str(gt_path))
        data  = img.get_fdata(dtype=np.float32)
        if data.ndim != 4:
            continue
        ed_f, es_f = _find_mnms_ed_es_frames(data)
        yield dict(patient_id=gt_path.stem.replace('.nii','').replace('_gt',''),
                   gt_path=gt_path, ed_frame=ed_f, es_frame=es_f,
                   affine=img.affine, source='mnms')


# ──────────────────────────────────────────────────────────────
# M&Ms-2 scanner — per-frame NIfTI layout
#
# Expected structure:
#   dataset/001/001_SA_ES_gt.nii   ← short-axis ES ground truth (used)
#   dataset/001/001_SA_ED_gt.nii
#   dataset/001/001_LA_ES_gt.nii   ← long-axis (ignored)
#   dataset/001/001_LA_ED_gt.nii
#
# Uses SA (short-axis) only — standard for AHA 17-segment WT.
# Files are already per-frame (3D), so no ED/ES auto-detection needed.
# ──────────────────────────────────────────────────────────────
def scan_mnms2(mnms2_dir):
    mnms2_dir = Path(mnms2_dir)
    for pat_dir in sorted(mnms2_dir.iterdir()):
        if not pat_dir.is_dir():
            continue

        # Find short-axis ES ground truth (handles .nii and .nii.gz)
        sa_es_files = (
            sorted(pat_dir.glob('*_SA_ES_gt.nii.gz')) +
            sorted(pat_dir.glob('*_SA_ES_gt.nii'))
        )
        if not sa_es_files:
            continue

        es_gt = sa_es_files[0]
        # If the path is actually a directory (nested layout), resolve it
        if es_gt.is_dir():
            try:
                es_gt = resolve_nii_path(es_gt)
            except FileNotFoundError:
                continue

        yield dict(
            patient_id = f'mnms2_{pat_dir.name}',
            es_gt      = es_gt,
            source     = 'mnms2',
        )

print('✅ Dataset scanners ready (nested-directory layout supported)')

## 10 · Per-Patient ES Processing

In [ ]:
def _normalize_xyz(xyz, centroid=None, scale=None):
    """ED-cache normalisation: subtract endo centroid, divide by mean
    short-axis radius.  Identical formula to ``build-lv-cache``."""
    out = xyz.copy().astype(np.float32)
    if centroid is None:
        cxy      = out[:, :2].mean(0)
        centroid = np.array([cxy[0], cxy[1], out[:, 2].mean()],
                             dtype=np.float32)
    if scale is None:
        scale = float(np.linalg.norm(out[:, :2] - centroid[:2], axis=1)
                                .mean().clip(min=1e-3))
    out -= centroid
    out /= scale
    return out, centroid, float(scale)


def _slice_mesh_at_z(verts, faces, z, pts_per_ring=60):
    """Plane-mesh intersection at z=const → sorted ring (mm)."""
    crossings = []
    for ia, ib in [(0, 1), (1, 2), (2, 0)]:
        za = verts[faces[:, ia], 2]
        zb = verts[faces[:, ib], 2]
        mask = (za - z) * (zb - z) < 0
        if mask.sum() == 0:
            continue
        t  = (z - za[mask]) / (zb[mask] - za[mask])
        pa = verts[faces[mask, ia], :2]
        pb = verts[faces[mask, ib], :2]
        crossings.append(pa + t[:, None] * (pb - pa))
    if not crossings:
        return None
    pts = np.vstack(crossings)
    if len(pts) < 3:
        return None
    c     = pts.mean(0)
    order = np.argsort(np.arctan2(pts[:, 1] - c[1], pts[:, 0] - c[0]))
    pts   = pts[order]
    if len(pts) > pts_per_ring:
        idx = np.round(np.linspace(0, len(pts) - 1, pts_per_ring)).astype(int)
        pts = pts[idx]
    return pts.astype(np.float32)


def synthesise_sax_contour(endo_v_n, epi_v_n, faces,
                            n_slices=10, pts_per_ring=60):
    """Build the ED-style ``contour (N, 4)`` array by slicing the
    NORMALISED endo + epi meshes at evenly-spaced z planes.

    Returns
    -------
    contour      : (N, 4) float32  — xyz + tissue (0=endo, 1=epi)
    slice_ids    : (N,)   int64
    slice_z      : (n_slices,) float32  — normalised z-centres
    slice_z_mask : (n_slices,) bool     — valid slices (both rings present)
    """
    all_v = np.vstack([endo_v_n, epi_v_n])
    z_min, z_max = all_v[:, 2].min(), all_v[:, 2].max()
    margin = (z_max - z_min) * 0.05
    z_ctrs = np.linspace(z_min + margin, z_max - margin, n_slices).astype(np.float32)

    xyz_chunks, tissue_chunks, sid_chunks = [], [], []
    valid = np.zeros(n_slices, dtype=bool)

    for k, zc in enumerate(z_ctrs):
        endo_ring = _slice_mesh_at_z(endo_v_n, faces, float(zc), pts_per_ring)
        epi_ring  = _slice_mesh_at_z(epi_v_n,  faces, float(zc), pts_per_ring)
        if endo_ring is None or epi_ring is None:
            continue
        valid[k] = True
        for ring, label in [(endo_ring, 0.0), (epi_ring, 1.0)]:
            n = len(ring)
            xyz_chunks.append(np.column_stack([ring, np.full(n, zc, np.float32)]))
            tissue_chunks.append(np.full(n, label, np.float32))
            sid_chunks.append(np.full(n, k, np.int64))

    if not xyz_chunks:
        return None, None, z_ctrs, valid

    contour = np.column_stack([
        np.vstack(xyz_chunks).astype(np.float32),
        np.concatenate(tissue_chunks),
    ]).astype(np.float32)
    slice_ids = np.concatenate(sid_chunks).astype(np.int64)
    return contour, slice_ids, z_ctrs, valid


def acdc_patient_contours(record, lbl_lv=3, lbl_myo=2,
                           n_pts=40, n_slices=10):
    """Return ``(endo_contour_pts, epi_contour_pts_or_None)``.

    Used by both ACDC and M&Ms-2 (per-frame SA NIfTIs).
    """
    seg, affine = load_nifti_frame(record["es_gt"], frame=0)
    return extract_lv_contours(seg, affine, lbl_lv, lbl_myo, n_pts, n_slices)


def mnms_patient_contours(record, lbl_lv=3, lbl_myo=2,
                           n_pts=40, n_slices=10):
    """Return ``(endo_contour_pts, epi_contour_pts_or_None)`` for a 4-D M&Ms volume."""
    img  = nib.load(str(record["gt_path"]))
    data = img.get_fdata(dtype=np.float32).astype(np.int32)
    seg  = data[..., record["es_frame"]]
    return extract_lv_contours(seg, record["affine"], lbl_lv, lbl_myo, n_pts, n_slices)


print("✅ Processing helpers ready (real-epi contour + ED-format synthesiser)")


## 11 · Build ES Occupancy Cache

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

sample_idx = 0
failed     = 0
meta       = []
_lock      = threading.Lock()

NUM_GPUS        = max(1, torch.cuda.device_count())
WORKERS_PER_GPU = 2     # safe — SSM tensors are cached once, not duplicated
MAX_WORKERS     = NUM_GPUS * WORKERS_PER_GPU
print(f"Using {MAX_WORKERS} workers across {NUM_GPUS} GPU(s)")

# ── Pre-warm: upload SSM tensors to each GPU ONCE ────────────
for g in range(NUM_GPUS):
    dev = f"cuda:{g}" if torch.cuda.is_available() else "cpu"
    _get_ssm_cached(dev, SSM_MEAN_VERTS, SSM_MODES, SSM_STDS, CFG["n_modes"])
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated(g) / 1e6
        total = torch.cuda.get_device_properties(g).total_memory / 1e6
        print(f"  GPU {g}:  SSM cached {alloc:.0f} MB / {total:.0f} MB total")

# ── Per-GPU CUDA streams for true kernel overlap ─────────────
_GPU_STREAMS = {}
if torch.cuda.is_available():
    for g in range(NUM_GPUS):
        _GPU_STREAMS[g] = [torch.cuda.Stream(device=g)
                           for _ in range(WORKERS_PER_GPU)]


# ─────────────────────────────────────────────────────────────
# Counters for diagnostics — printed once per dataset
# ─────────────────────────────────────────────────────────────
_epi_source_stats = {"real": 0, "normal_offset": 0}


def _derive_epi_from_real_contour(endo_verts, faces, epi_pts,
                                  fallback_offset=4.0):
    """Per-slice radial expansion of the fitted endo SSM mesh onto the
    REAL epicardial SAX contour.

    The endo SSM topology is preserved one-to-one (same vertex count,
    same faces) so all downstream model dimensions match ED exactly.
    For every endo vertex we:
      • find the nearest epi-contour ring in z;
      • use the ring centroid (x,y) as the cavity centre;
      • snap the vertex (x,y) to the ring sample whose angular
        position best matches the endo vertex angle.
    Apex / out-of-band vertices fall back to a small oriented-normal
    offset.  Returns ``None`` when no usable real contour is given.
    """
    import numpy as _np
    if epi_pts is None or len(epi_pts) < 10:
        return None
    epi_pts = _np.asarray(epi_pts, dtype=_np.float64)

    # Group ring samples by z (rounded to mm precision)
    z_round = _np.round(epi_pts[:, 2], 3)
    unique_z = _np.sort(_np.unique(z_round))
    if len(unique_z) < 2:
        return None
    rings = [epi_pts[z_round == z] for z in unique_z]
    centers = _np.stack([r[:, :2].mean(0) for r in rings])  # (K, 2)
    dz = float(_np.median(_np.diff(unique_z)))
    band = max(dz * 0.6, 1.0)

    endo_verts = _np.asarray(endo_verts, dtype=_np.float64)
    epi_verts = endo_verts.copy()

    # Nearest ring per endo vertex
    z_v = endo_verts[:, 2]
    z_diff = _np.abs(z_v[:, None] - unique_z[None, :])
    nearest_k = _np.argmin(z_diff, axis=1)
    in_band = z_diff[_np.arange(len(z_v)), nearest_k] <= band

    # In-band: project to ring (preserve endo z)
    for k in range(len(unique_z)):
        sel = (nearest_k == k) & in_band
        if not sel.any():
            continue
        ring = rings[k]
        cx, cy = centers[k]
        ev = endo_verts[sel]
        theta_v = _np.arctan2(ev[:, 1] - cy, ev[:, 0] - cx)
        theta_r = _np.arctan2(ring[:, 1] - cy, ring[:, 0] - cx)
        # Cyclic nearest-angle lookup
        d = _np.angle(_np.exp(1j * (theta_r[None, :] - theta_v[:, None])))
        j = _np.argmin(_np.abs(d), axis=1)
        epi_verts[sel, 0] = ring[j, 0]
        epi_verts[sel, 1] = ring[j, 1]
        epi_verts[sel, 2] = ev[:, 2]

    # Out-of-band (apex/base extremes): tapered oriented-normal offset
    if (~in_band).any():
        try:
            normals = compute_vertex_normals(endo_verts, faces,
                                             orient_outward=True)
            epi_verts[~in_band] = (endo_verts[~in_band]
                                   + normals[~in_band] * fallback_offset)
        except Exception:
            # Fallback: radial outward from global centroid
            c = endo_verts.mean(0)
            d = endo_verts[~in_band] - c
            n = d / (_np.linalg.norm(d, axis=1, keepdims=True) + 1e-9)
            epi_verts[~in_band] = endo_verts[~in_band] + n * fallback_offset

    return epi_verts.astype(_np.float32)


def _process_one(args):
    """Full pipeline per patient — runs entirely inside the thread pool.

    Steps:
      1. Extract endo + (optional real) epi contours from the NIfTI.
      2. Fit SSM to endo  → ``endo_verts``.
      3. Derive epi:
           • if a real epi contour exists, fit SSM to it → ``epi_verts``;
           • otherwise call ``generate_epi`` (oriented normal offset).
      4. Sample occupancy with the DUAL-MASK schema.
      5. Normalise, synthesise SAX contour, write ED-format ``.npz``.
    """
    global sample_idx, failed
    rec, get_contours_fn, worker_id, lbl_lv, lbl_myo, pts_per_ring, n_sax_slices = args
    gpu_id = worker_id % NUM_GPUS
    device = f"cuda:{gpu_id}" if torch.cuda.is_available() else "cpu"
    pid    = rec.get("patient_id", "?")

    stream_ctx = nullcontext()
    if gpu_id in _GPU_STREAMS:
        s_idx = (worker_id // NUM_GPUS) % len(_GPU_STREAMS[gpu_id])
        stream_ctx = torch.cuda.stream(_GPU_STREAMS[gpu_id][s_idx])

    # ── 1. Contour extraction (CPU) ──────────────────────────
    try:
        endo_pts, epi_pts = get_contours_fn(
            rec, lbl_lv=lbl_lv, lbl_myo=lbl_myo,
            n_pts=pts_per_ring, n_slices=n_sax_slices,
        )
    except Exception as e:
        return pid, False, f"contour error: {e}"

    if endo_pts is None or len(endo_pts) < 20:
        return pid, False, "too few endo contour points"

    # ── 2. Allocate a unique sample index (thread-safe) ──────
    with _lock:
        idx = sample_idx
        sample_idx += 1

    # ── 3. SSM fitting on endo ───────────────────────────────
    try:
        with stream_ctx:
            endo_verts, coeffs, loss = fit_ssm_to_contours(
                endo_pts, SSM_MEAN_VERTS, SSM_MODES, SSM_STDS,
                n_modes=CFG["n_modes"],
                regularise=CFG["ssm_regularise_es"],
                max_iter=CFG["ssm_max_iter"],
                device=device,
            )
    except Exception as e:
        return pid, False, f"endo SSM fit failed: {e}"

    # ── 4. Epi derivation: per-slice radial expansion onto real contour
    #     → fallback to oriented-normal offset when no real contour.
    epi_source = "real"
    epi_verts  = _derive_epi_from_real_contour(endo_verts, SSM_FACES, epi_pts)
    if epi_verts is None:
        epi_source = "normal_offset"
        epi_verts  = generate_epi(
            endo_verts, SSM_FACES,
            base_offset=CFG["epi_base_offset"],
            apex_offset=CFG["epi_apex_offset"],
            noise_std=CFG["epi_noise_std"],
            seed=idx,
        )
    else:
        # Sanity: ensure expanded epi truly encloses endo.
        cm     = 0.5 * (endo_verts.mean(0) + epi_verts.mean(0))
        r_endo = np.linalg.norm(endo_verts - cm, axis=1).mean()
        r_epi  = np.linalg.norm(epi_verts  - cm, axis=1).mean()
        if r_epi <= r_endo * 1.02:
            epi_source = "normal_offset"
            epi_verts  = generate_epi(
                endo_verts, SSM_FACES,
                base_offset=CFG["epi_base_offset"],
                apex_offset=CFG["epi_apex_offset"],
                noise_std=CFG["epi_noise_std"],
                seed=idx,
            )
    with _lock:
        _epi_source_stats[epi_source] += 1

    # ── 5. Occupancy sampling — TWO MASKS ────────────────────
    try:
        pts_w, endo_occ, epi_occ = sample_occupancy(
            endo_verts, epi_verts, SSM_FACES,
            n_query=CFG["n_query"],
            surface_std=CFG["surface_std"],
            seed=idx,
        )
    except Exception as e:
        return pid, False, f"occupancy failed: {e}"

    # ── 6. Normalise + synthesise SAX contour (ED schema) ────
    _, centroid, scale = _normalize_xyz(endo_verts.astype(np.float32))
    endo_v_n, _, _ = _normalize_xyz(endo_verts.astype(np.float32),
                                     centroid, scale)
    epi_v_n,  _, _ = _normalize_xyz(epi_verts.astype(np.float32),
                                     centroid, scale)
    query_n = ((pts_w - centroid) / scale).astype(np.float32)

    contour, slice_ids, slice_z, slice_z_mask = synthesise_sax_contour(
        endo_v_n, epi_v_n, SSM_FACES,
        n_slices=CFG["n_sax_slices"],
        pts_per_ring=CFG["pts_per_ring"],
    )
    if contour is None or slice_z_mask.sum() < 3:
        return pid, False, "could not synthesise SAX contour"

    # ── 7. Save in ED-compatible schema ──────────────────────
    out_path = Path(CFG["es_cache_dir"]) / f"sample_{idx:04d}.npz"
    np.savez_compressed(
        str(out_path),
        contour       = contour,
        slice_ids     = slice_ids.astype(np.int64),
        slice_z       = slice_z.astype(np.float32),
        slice_z_mask  = slice_z_mask.astype(np.bool_),
        query         = query_n,
        endo_occ      = endo_occ,
        epi_occ       = epi_occ,
        endo_vertices = endo_v_n.astype(np.float32),
        endo_faces    = SSM_FACES.astype(np.int32),
        epi_vertices  = epi_v_n.astype(np.float32),
        epi_faces     = SSM_FACES.astype(np.int32),
        scale         = np.float32(scale),
        centroid      = centroid.astype(np.float32),
        coeffs        = coeffs.astype(np.float32),
        ssm_fit_loss  = np.float32(loss),
        phase         = "ES",
        epi_source    = epi_source,
    )

    with _lock:
        meta.append(dict(idx=idx, source=rec.get("source", ""),
                          patient_id=pid, epi_source=epi_source))

    return pid, True, None


def _run_dataset(records, get_contours_fn, label):
    global failed

    records = list(records)
    if TRIAL_MODE:
        records = records[:2]
    print(f"\n--- {label}: {len(records)} patients ---")

    args_list = [
        (rec, get_contours_fn, i,
         CFG["lbl_lv"], CFG["lbl_myo"],
         CFG["pts_per_ring"], CFG["n_sax_slices"])
        for i, rec in enumerate(records)
    ]

    from tqdm.notebook import tqdm as tqdm_nb
    pbar = tqdm_nb(total=len(records), desc=label)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futs = {pool.submit(_process_one, a): a for a in args_list}
        done_count = 0
        for fut in as_completed(futs):
            pid, ok, err = fut.result()
            if not ok:
                print(f"  ✗ {pid}  {err}")
                with _lock:
                    failed += 1
            done_count += 1
            if done_count % 10 == 0:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            pbar.update(1)
    pbar.close()

    if torch.cuda.is_available():
        for g in range(NUM_GPUS):
            alloc = torch.cuda.memory_allocated(g) / 1e6
            peak  = torch.cuda.max_memory_allocated(g) / 1e6
            print(f"  GPU {g}  current={alloc:.0f} MB  peak={peak:.0f} MB")
        torch.cuda.empty_cache()
        gc.collect()


# ── ACDC ─────────────────────────────────────────────────────
if CFG["acdc_dir"]:
    _run_dataset(scan_acdc(CFG["acdc_dir"]), acdc_patient_contours, "ACDC")

# ── M&Ms ─────────────────────────────────────────────────────
if CFG["mnms_dir"]:
    _run_dataset(scan_mnms(CFG["mnms_dir"]), mnms_patient_contours, "M&Ms")

# ── M&Ms-2 (per-frame SA files → same contour fn as ACDC) ───
if CFG["mnms2_dir"]:
    _run_dataset(scan_mnms2(CFG["mnms2_dir"]), acdc_patient_contours, "M&Ms-2")

print(f"\n✅ ES cache built: {sample_idx} samples  |  {failed} failed")
print(f"   epi sources — real: {_epi_source_stats['real']}  "
      f"normal-offset fallback: {_epi_source_stats['normal_offset']}")

# ── Save split.json (80 / 10 / 10 — matches ED cache) ────────
all_idx = list(range(sample_idx))
np.random.seed(CFG["seed"])
np.random.shuffle(all_idx)
n_tr  = int(0.80 * len(all_idx))
n_val = int(0.10 * len(all_idx))
split = dict(tr=all_idx[:n_tr],
             val=all_idx[n_tr:n_tr + n_val],
             te=all_idx[n_tr + n_val:])
with open(Path(CFG["es_cache_dir"]) / "split.json", "w") as f:
    json.dump(split, f)
print(f"split.json  tr:{len(split['tr'])}  val:{len(split['val'])}  te:{len(split['te'])}")

# ── Sanity check on the freshly written cache ────────────────
es_files = sorted(Path(CFG["es_cache_dir"]).glob("sample_*.npz"))
if es_files:
    rng_chk = np.random.default_rng(0)
    pick    = rng_chk.choice(len(es_files), size=min(8, len(es_files)),
                              replace=False)
    print("\n── Sanity check (8 random samples) ──")
    print(f"  {'idx':>4}  {'endo%':>6}  {'epi%':>6}  {'wall%':>6}  "
          f"{'r_endo':>7}  {'r_epi':>7}  {'src':>14}")
    bad = 0
    for k in pick:
        d = np.load(es_files[int(k)], allow_pickle=True)
        e_frac = float(d["endo_occ"].mean())
        p_frac = float(d["epi_occ"].mean())
        w_frac = float(((d["epi_occ"] > 0.5) & (d["endo_occ"] < 0.5)).mean())
        r_e    = float(np.linalg.norm(d["endo_vertices"], axis=1).mean())
        r_p    = float(np.linalg.norm(d["epi_vertices"],  axis=1).mean())
        src    = str(d.get("epi_source", "?"))
        ok     = (p_frac > e_frac + 0.02) and (w_frac > 0.02) and (r_p > r_e * 1.02)
        if not ok:
            bad += 1
        print(f"  {int(k):>4}  {e_frac*100:>5.1f}%  {p_frac*100:>5.1f}%  "
              f"{w_frac*100:>5.1f}%  {r_e:>7.3f}  {r_p:>7.3f}  {src:>14}"
              f"{'  ⚠' if not ok else ''}")
    if bad == 0:
        print("  ✅ all sampled cells satisfy: epi > endo, wall > 2%, r_epi > 1.02·r_endo")
    else:
        print(f"  ⚠ {bad}/{len(pick)} samples failed the sanity test — inspect above")


## 12 · Quick Visualisation of ES Samples

In [ ]:
es_files = sorted(Path(CFG['es_cache_dir']).glob('sample_*.npz'))[:4]
if not es_files:
    print('No ES samples found — run cell 11 first.')
else:
    fig, axes = plt.subplots(1, len(es_files), figsize=(4 * len(es_files), 4))
    if len(es_files) == 1:
        axes = [axes]
    for ax, fp in zip(axes, es_files):
        d = np.load(fp, allow_pickle=True)
        # New schema: ED-compatible keys.  Fall back to the old ones for
        # any legacy caches that may still be on disk.
        if 'endo_vertices' in d.files:
            endo = d['endo_vertices']; epi = d['epi_vertices']
        else:
            endo = d['endo_verts'];    epi = d['epi_verts']
        # Project onto first two PCs of endo for a 2-D view
        _, _, vt = np.linalg.svd(endo - endo.mean(0), full_matrices=False)
        proj_e  = endo @ vt[:2].T
        proj_ep = epi  @ vt[:2].T
        ax.scatter(proj_e[:, 0],  proj_e[:, 1],  s=1, c='steelblue', label='endo')
        ax.scatter(proj_ep[:, 0], proj_ep[:, 1], s=1, c='coral',     label='epi')
        ax.set_title(f"{fp.stem}  src={str(d.get('epi_source','?'))}", fontsize=8)
        ax.set_aspect('equal'); ax.axis('off')
    axes[0].legend(markerscale=5, fontsize=7)
    plt.suptitle('ES mesh projections (first 2 PCA dims)', fontsize=10)
    plt.tight_layout()
    plt.show()
    print(f'✅ Visualised {len(es_files)} ES samples')


## 12b · Interactive 3D Mesh Visualisation

Four panels per sample:
1. **Endo + Epi overlay** — semi-transparent epi wrapping the endo surface
2. **AHA 17-segment map** — each endo vertex coloured by its AHA segment
3. **Wall-thickness heatmap** — endo coloured by per-vertex WT (mm)
4. **Epi surface only** — for inspection

Uses **Plotly** for in-notebook rotation / zoom / pan.

In [ ]:
%pip install -q plotly

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# ── AHA segment colour palette (17 distinct colours) ─────────
_AHA_COLORS = {
    1: '#e6194b', 2: '#3cb44b', 3: '#ffe119', 4: '#4363d8',
    5: '#f58231', 6: '#911eb4', 7: '#42d4f4', 8: '#f032e6',
    9: '#bfef45',10: '#fabed4',11: '#469990',12: '#dcbeff',
   13: '#9A6324',14: '#fffac8',15: '#800000',16: '#aaffc3',
   17: '#808000',
}
_AHA_NAMES = {
    1:'Basal anterior',       2:'Basal anteroseptal',
    3:'Basal inferoseptal',   4:'Basal inferior',
    5:'Basal inferolateral',  6:'Basal anterolateral',
    7:'Mid anterior',         8:'Mid anteroseptal',
    9:'Mid inferoseptal',    10:'Mid inferior',
   11:'Mid inferolateral',   12:'Mid anterolateral',
   13:'Apical anterior',     14:'Apical septal',
   15:'Apical inferior',     16:'Apical lateral',
   17:'Apex',
}


def _mesh3d(verts, faces, color_values=None, colorscale=None,
            flat_color=None, opacity=1.0, name='', showscale=False,
            cmin=None, cmax=None, colorbar_title='', hovertext=None):
    """Helper: build a Plotly Mesh3d trace."""
    kw = dict(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        opacity=opacity, name=name,
        flatshading=False,
        lighting=dict(ambient=0.5, diffuse=0.7, specular=0.3,
                      roughness=0.4, fresnel=0.2),
        lightposition=dict(x=100, y=200, z=300),
    )
    if color_values is not None:
        kw['intensity'] = color_values
        kw['colorscale'] = colorscale or 'YlOrRd'
        kw['showscale'] = showscale
        kw['cmin'] = cmin
        kw['cmax'] = cmax
        if showscale:
            kw['colorbar'] = dict(title=colorbar_title, thickness=15, len=0.6)
    elif flat_color:
        kw['color'] = flat_color
    if hovertext is not None:
        kw['hovertext'] = hovertext
        kw['hoverinfo'] = 'text'
    return go.Mesh3d(**kw)


def _camera(verts):
    """Default camera looking at the mesh centroid."""
    c = verts.mean(0)
    span = np.ptp(verts, axis=0).max()
    return dict(
        eye=dict(x=0, y=-2.0, z=0.6),
        center=dict(x=0, y=0, z=0),
        up=dict(x=0, y=0, z=1),
    )


def _layout3d(**kw):
    return dict(
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='data',
            bgcolor='#fafafa',
        ),
        margin=dict(l=0, r=0, t=40, b=0),
        **kw,
    )


# ──────────────────────────────────────────────────────────────
# Main visualisation function
# ──────────────────────────────────────────────────────────────
def visualise_3d_sample(npz_path, show=True):
    """
    Build an interactive 3D figure for a single cached sample.

    Returns the Plotly figure.
    """
    d       = np.load(npz_path)
    endo    = d['endo_verts'].astype(np.float64)
    epi     = d['epi_verts'].astype(np.float64)
    faces   = d['faces'].astype(np.int32)
    phase   = str(d.get('phase', '?'))
    loss    = float(d.get('ssm_fit_loss', np.nan))

    apex_idx, _, base_center = find_apex_base(endo, faces)
    wt          = compute_wall_thickness(endo, epi)
    seg_labels  = assign_aha_segments(endo, apex_idx, base_center)

    # ── Panel 1: Endo + Epi overlay ──────────────────────────
    fig1 = go.Figure()
    fig1.add_trace(_mesh3d(endo, faces, flat_color='steelblue',
                           opacity=1.0, name='Endo'))
    fig1.add_trace(_mesh3d(epi, faces, flat_color='coral',
                           opacity=0.25, name='Epi'))
    fig1.update_layout(
        title=dict(text=f'Endo + Epi overlay  ({phase}, loss={loss:.3f})',
                   font=dict(size=13)),
        scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False),
                   zaxis=dict(visible=False), aspectmode='data',
                   bgcolor='#fafafa'),
        scene_camera=_camera(endo),
        margin=dict(l=0, r=0, t=40, b=0),
        legend=dict(x=0.01, y=0.99),
        width=700, height=550,
    )

    # ── Panel 2: AHA segment map ─────────────────────────────
    seg_colors = np.array([list(int(
        _AHA_COLORS[seg_labels[i]].lstrip('#')[j:j+2], 16)
        for j in (0, 2, 4)) for i in range(len(endo))], dtype=np.float32) / 255.0
    hover_seg = [f'S{seg_labels[i]}: {_AHA_NAMES[seg_labels[i]]}<br>WT={wt[i]:.1f} mm'
                 for i in range(len(endo))]
    fig2 = go.Figure()
    fig2.add_trace(_mesh3d(
        endo, faces,
        color_values=seg_labels.astype(float),
        colorscale=[
            [i/16, _AHA_COLORS[i+1]] for i in range(17)
        ],
        cmin=1, cmax=17,
        showscale=True, colorbar_title='AHA Seg',
        opacity=1.0, name='AHA segments',
        hovertext=hover_seg,
    ))
    fig2.update_layout(
        title=dict(text=f'AHA 17-Segment Map  ({phase})', font=dict(size=13)),
        scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False),
                   zaxis=dict(visible=False), aspectmode='data',
                   bgcolor='#fafafa'),
        scene_camera=_camera(endo),
        margin=dict(l=0, r=0, t=40, b=0),
        width=700, height=550,
    )

    # ── Panel 3: Wall-thickness heatmap ──────────────────────
    hover_wt = [f'WT={wt[i]:.2f} mm<br>S{seg_labels[i]}: {_AHA_NAMES[seg_labels[i]]}'
                for i in range(len(endo))]
    fig3 = go.Figure()
    fig3.add_trace(_mesh3d(
        endo, faces,
        color_values=wt,
        colorscale='YlOrRd',
        cmin=max(0, np.percentile(wt, 2)),
        cmax=np.percentile(wt, 98),
        showscale=True, colorbar_title='WT (mm)',
        opacity=1.0, name='Wall thickness',
        hovertext=hover_wt,
    ))
    fig3.update_layout(
        title=dict(text=f'Wall Thickness Heatmap  ({phase})', font=dict(size=13)),
        scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False),
                   zaxis=dict(visible=False), aspectmode='data',
                   bgcolor='#fafafa'),
        scene_camera=_camera(endo),
        margin=dict(l=0, r=0, t=40, b=0),
        width=700, height=550,
    )

    # ── Panel 4: Epi surface alone (wireframe-style) ─────────
    fig4 = go.Figure()
    fig4.add_trace(_mesh3d(epi, faces, flat_color='#ff7f50',
                           opacity=0.85, name='Epi surface'))
    # Add endo as ghost for reference
    fig4.add_trace(_mesh3d(endo, faces, flat_color='#4682b4',
                           opacity=0.15, name='Endo (ghost)'))
    fig4.update_layout(
        title=dict(text=f'Epi Surface  ({phase})', font=dict(size=13)),
        scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False),
                   zaxis=dict(visible=False), aspectmode='data',
                   bgcolor='#fafafa'),
        scene_camera=_camera(endo),
        margin=dict(l=0, r=0, t=40, b=0),
        legend=dict(x=0.01, y=0.99),
        width=700, height=550,
    )

    if show:
        for f in [fig1, fig2, fig3, fig4]:
            f.show()

    return fig1, fig2, fig3, fig4


# ── Run on first 2 cached ES samples ─────────────────────────
es_files = sorted(Path(CFG['es_cache_dir']).glob('sample_*.npz'))
if not es_files:
    print('No ES cache files found — run cell 11 first.')
else:
    for fp in es_files[:2]:
        print(f'\n{"═"*60}')
        print(f'  {fp.name}')
        print(f'{"═"*60}')
        _ = visualise_3d_sample(fp)
    print(f'\n✅ Interactive 3D visualisation shown for {min(2, len(es_files))} samples')

In [ ]:
# ──────────────────────────────────────────────────────────────
# Cross-section slicer — animated sweep through the LV
# ──────────────────────────────────────────────────────────────
def visualise_cross_sections(npz_path, n_slices=8):
    """
    Show axial cross-section rings through the endo/epi mesh
    with WT annotations, plus a 3D overview with slice planes.
    """
    d     = np.load(npz_path)
    endo  = d['endo_verts'].astype(np.float64)
    epi   = d['epi_verts'].astype(np.float64)
    faces = d['faces'].astype(np.int32)
    phase = str(d.get('phase', '?'))

    apex_idx, _, base_center = find_apex_base(endo, faces)
    apex   = endo[apex_idx]
    long_v = base_center - apex
    long_l = np.linalg.norm(long_v) + 1e-8
    long_u = long_v / long_l

    # Longitudinal coordinate per vertex
    t_endo = np.dot(endo - apex, long_u) / long_l
    t_epi  = np.dot(epi  - apex, long_u) / long_l

    wt = compute_wall_thickness(endo, epi)

    # ── 2D cross-section grid ────────────────────────────────
    slice_t = np.linspace(0.05, 0.95, n_slices)
    ncols   = min(4, n_slices)
    nrows   = int(np.ceil(n_slices / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
    axes = np.atleast_2d(axes)

    # Project onto plane perpendicular to long axis
    # Local x/y = first two components of SVD of radial vectors
    radial_endo = endo - (apex + t_endo[:, None] * long_v)
    _, _, vt = np.linalg.svd(radial_endo, full_matrices=False)
    e1, e2 = vt[0], vt[1]

    for si, t_target in enumerate(slice_t):
        row, col = divmod(si, ncols)
        ax = axes[row, col]

        # Select vertices in a ± bandwidth slab
        bw = 0.5 / n_slices
        mask_en = np.abs(t_endo - t_target) < bw
        mask_ep = np.abs(t_epi  - t_target) < bw

        if mask_en.sum() < 5:
            ax.set_title(f't={t_target:.2f}\n(empty)', fontsize=8)
            ax.axis('off')
            continue

        # Project to local 2D
        en2d = np.column_stack([endo[mask_en] @ e1, endo[mask_en] @ e2])
        ep2d = np.column_stack([epi[mask_ep]  @ e1, epi[mask_ep]  @ e2])
        wt_slice = wt[mask_en]

        sc = ax.scatter(en2d[:, 0], en2d[:, 1], c=wt_slice, s=3,
                        cmap='YlOrRd', vmin=0, vmax=np.percentile(wt, 98))
        ax.scatter(ep2d[:, 0], ep2d[:, 1], s=1, c='grey', alpha=0.4, label='epi')

        level = 'Apex' if t_target < 0.15 else ('Apical' if t_target < 0.35 else
                ('Mid' if t_target < 0.68 else 'Basal'))
        ax.set_title(f'{level}  (t={t_target:.2f})\nmean WT={wt_slice.mean():.1f} mm',
                     fontsize=8)
        ax.set_aspect('equal')
        ax.axis('off')

    # Remove unused axes
    for si in range(n_slices, nrows * ncols):
        row, col = divmod(si, ncols)
        axes[row, col].axis('off')

    plt.suptitle(f'Cross-section slices — {Path(npz_path).stem} ({phase})',
                 fontsize=11, y=1.01)
    plt.tight_layout()
    plt.colorbar(sc, ax=axes.ravel().tolist(), label='WT (mm)',
                 fraction=0.02, pad=0.02)
    plt.show()

    # ── 3D overview with coloured slice bands ────────────────
    fig3d = go.Figure()
    # Full mesh (ghost)
    fig3d.add_trace(_mesh3d(endo, faces, flat_color='steelblue',
                            opacity=0.12, name='Endo'))

    colours = plt.cm.tab10(np.linspace(0, 1, n_slices))
    for si, t_target in enumerate(slice_t):
        bw = 0.5 / n_slices
        mask = np.abs(t_endo - t_target) < bw
        if mask.sum() < 5:
            continue
        c_hex = '#{:02x}{:02x}{:02x}'.format(
            int(colours[si][0]*255), int(colours[si][1]*255), int(colours[si][2]*255))
        level = 'Apex' if t_target < 0.15 else ('Apical' if t_target < 0.35 else
                ('Mid' if t_target < 0.68 else 'Basal'))
        fig3d.add_trace(go.Scatter3d(
            x=endo[mask, 0], y=endo[mask, 1], z=endo[mask, 2],
            mode='markers', marker=dict(size=1.5, color=c_hex),
            name=f'{level} t={t_target:.2f}',
            hovertext=[f'WT={wt[i]:.1f}mm' for i in np.where(mask)[0]],
        ))
    fig3d.update_layout(
        title=dict(text=f'Slice locations on endo — {Path(npz_path).stem}',
                   font=dict(size=12)),
        scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False),
                   zaxis=dict(visible=False), aspectmode='data',
                   bgcolor='#fafafa'),
        scene_camera=_camera(endo),
        margin=dict(l=0, r=0, t=40, b=0),
        width=700, height=500,
    )
    fig3d.show()
    print(f'✅ Cross-sections shown for {Path(npz_path).stem}')


# ── Run on first sample ──────────────────────────────────────
es_files = sorted(Path(CFG['es_cache_dir']).glob('sample_*.npz'))
if es_files:
    visualise_cross_sections(es_files[0], n_slices=8)
else:
    print('No ES cache files — run cell 11 first.')

In [ ]:
# ──────────────────────────────────────────────────────────────
# Population-level WT box plot + radar chart per AHA segment
# ──────────────────────────────────────────────────────────────
es_files = sorted(Path(CFG['es_cache_dir']).glob('sample_*.npz'))

if len(es_files) >= 3:
    # Collect per-segment WT across all patients
    seg_wt = {s: [] for s in range(1, 18)}
    for fp in tqdm(es_files, desc='Aggregating WT'):
        try:
            d     = np.load(fp)
            endo  = d['endo_verts'].astype(np.float64)
            epi   = d['epi_verts'].astype(np.float64)
            faces = d['faces'].astype(np.int32)
            wt_map, _, _ = aha_wall_thickness_map(endo, epi, faces)
            for s in range(1, 18):
                if s in wt_map and not np.isnan(wt_map[s]):
                    seg_wt[s].append(wt_map[s])
        except Exception:
            pass

    # ── Box plot ─────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5),
                                    gridspec_kw={'width_ratios': [3, 2]})

    ring_colors = {
        'Basal': '#4363d8', 'Mid': '#3cb44b',
        'Apical': '#f58231', 'Apex': '#e6194b',
    }
    box_data, labels, colors = [], [], []
    for s in range(1, 18):
        vals = seg_wt[s]
        if vals:
            box_data.append(vals)
            labels.append(f'S{s}')
            ring = _RING_NAME[s]
            colors.append(ring_colors[ring])

    bp = ax1.boxplot(box_data, labels=labels, patch_artist=True,
                     widths=0.6, showfliers=False)
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.65)
    for median in bp['medians']:
        median.set_color('black')
        median.set_linewidth(1.5)

    ax1.set_ylabel('Wall Thickness (mm)', fontsize=10)
    ax1.set_xlabel('AHA Segment', fontsize=10)
    ax1.set_title('Per-Segment WT Distribution (all patients)', fontsize=11)
    ax1.grid(axis='y', alpha=0.3)

    # Legend for rings
    for ring, c in ring_colors.items():
        ax1.bar(0, 0, color=c, alpha=0.65, label=ring)
    ax1.legend(loc='upper right', fontsize=8)

    # ── Radar chart of mean WT ───────────────────────────────
    means = [np.mean(seg_wt[s]) if seg_wt[s] else 0 for s in range(1, 18)]
    stds  = [np.std(seg_wt[s])  if seg_wt[s] else 0 for s in range(1, 18)]
    angles = np.linspace(0, 2 * np.pi, 17, endpoint=False).tolist()
    means_c = means + [means[0]]
    upper_c = [m + sd for m, sd in zip(means, stds)] + [means[0] + stds[0]]
    lower_c = [max(0, m - sd) for m, sd in zip(means, stds)] + [max(0, means[0] - stds[0])]
    angles_c = angles + [angles[0]]
    seg_names = [f'S{s}' for s in range(1, 18)]

    ax2 = plt.subplot(122, polar=True)
    ax2.fill_between(angles_c, lower_c, upper_c, alpha=0.2, color='steelblue')
    ax2.plot(angles_c, means_c, 'o-', color='steelblue', linewidth=1.5,
             markersize=4, label='Mean ± 1σ')
    ax2.set_xticks(angles)
    ax2.set_xticklabels(seg_names, fontsize=7)
    ax2.set_title('Mean WT by AHA Segment', fontsize=11, pad=20)
    ax2.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=8)

    plt.suptitle(f'Population WT Summary  (n={len(es_files)} ES samples)',
                 fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

    # ── Print expected-range check ───────────────────────────
    print('\n── Physiological range check ──')
    print('Expected ES WT (literature): Basal 8-14 mm, Mid 7-12 mm, Apical 5-10 mm')
    for ring, segs in [('Basal', range(1,7)), ('Mid', range(7,13)),
                       ('Apical', range(13,17)), ('Apex', [17])]:
        vals = []
        for s in segs:
            vals.extend(seg_wt[s])
        if vals:
            print(f'  {ring:8s}: {np.mean(vals):.1f} ± {np.std(vals):.1f} mm  '
                  f'(range {np.min(vals):.1f} – {np.max(vals):.1f})')
else:
    print('Need ≥ 3 ES samples for population summary. Run cell 11 first.')

## 13 · Verify ES Cache & Split

Quick sanity check on the ES occupancy cache: file count, split
distribution, and a few sample stats.

In [ ]:
ES_CACHE = Path(CFG['es_cache_dir'])
es_files = sorted(ES_CACHE.glob('sample_*.npz'))

# ── Verify split.json exists ──────────────────────────────────
split_path = ES_CACHE / 'split.json'
if split_path.exists():
    with open(split_path) as f:
        split = json.load(f)
    print(f'ES cache → {ES_CACHE}')
    print(f'  Files : {len(es_files)}')
    print(f'  Train : {len(split["tr"])}')
    print(f'  Val   : {len(split["val"])}')
    print(f'  Test  : {len(split["te"])}')
else:
    print('⚠  split.json missing — rebuilding …')
    all_idx = list(range(len(es_files)))
    np.random.seed(CFG['seed'])
    np.random.shuffle(all_idx)
    n_tr  = int(0.70 * len(all_idx))
    n_val = int(0.15 * len(all_idx))
    split = dict(tr=all_idx[:n_tr],
                 val=all_idx[n_tr:n_tr+n_val],
                 te=all_idx[n_tr+n_val:])
    with open(split_path, 'w') as f:
        json.dump(split, f)
    print(f'Rebuilt split.json  tr:{len(split["tr"])}  val:{len(split["val"])}  te:{len(split["te"])}')

# ── Quick stats on a few samples ──────────────────────────────
if es_files:
    losses = []
    for fp in es_files[:20]:
        d = np.load(fp)
        losses.append(float(d.get('ssm_fit_loss', np.nan)))
    print(f'\n  SSM fit loss (first {len(losses)}): '
          f'mean={np.nanmean(losses):.4f}  '
          f'min={np.nanmin(losses):.4f}  '
          f'max={np.nanmax(losses):.4f}')

print(f'\nTo retrain the INR on ES data, set:')
print(f'  CFG["cache_dir"] = "{ES_CACHE}"')

## 14 · AHA 17-Segment Wall Thickness

Once the INR has been retrained on the merged cache, it can produce
clean endo meshes at both ED and ES. This section computes wall thickness
for each of the 17 AHA segments directly on those meshes.

**Key properties**
- Both ED and ES meshes share the same SSM topology (same faces, same vertex count)
  so per-vertex correspondence is exact — no registration needed.
- Wall thickness per vertex = distance from endo vertex to nearest epi vertex.
- Segment boundaries follow the standard AHA 17-model:
  basal (1–6) · mid (7–12) · apical (13–16) · apex (17).


In [ ]:
# ─── AHA segment assignment ──────────────────────────────────
def assign_aha_segments(verts, apex_idx, base_center, anterior_ref=None):
    """
    Assign each endo vertex to one of the 17 AHA segments.

    Segment layout (AHA convention, viewed from apex toward base):
      Basal  (1–6)  : 6 × 60° sectors,  top 34% of LV length
      Mid    (7–12) : 6 × 60° sectors,  mid 33%
      Apical (13–16): 4 × 90° sectors,  next 23%
      Apex   (17)   : single segment,   bottom 10%

    Parameters
    ----------
    verts        : (N, 3) endo vertices (mm)
    apex_idx     : int — apex vertex index
    base_center  : (3,) — centroid of base ring
    anterior_ref : (3,) optional — direction defining θ=0 (anterior wall).
                   Computed from mid-zone PCA if not supplied.

    Returns
    -------
    seg : (N,) int32 with values 1..17
    """
    apex      = verts[apex_idx]
    long_vec  = base_center - apex
    long_len  = np.linalg.norm(long_vec) + 1e-8
    long_unit = long_vec / long_len

    # Longitudinal coordinate: 0 = apex, 1 = base
    proj = np.clip(np.dot(verts - apex, long_unit) / long_len, 0, 1)

    # Radial vectors (perpendicular to long axis)
    radial = verts - (apex + proj[:, None] * long_vec)

    # Anterior reference for θ = 0
    if anterior_ref is None:
        mid_mask = (proj > 0.33) & (proj < 0.67)
        if mid_mask.sum() >= 3:
            _, _, vt = np.linalg.svd(radial[mid_mask], full_matrices=False)
            anterior_ref = vt[0]
        else:
            candidate = np.array([1.0, 0.0, 0.0])
            if abs(np.dot(candidate, long_unit)) > 0.9:
                candidate = np.array([0.0, 1.0, 0.0])
            anterior_ref = np.cross(long_unit, candidate)
            anterior_ref /= np.linalg.norm(anterior_ref) + 1e-8

    lateral_ref = np.cross(long_unit, anterior_ref)
    lateral_ref /= np.linalg.norm(lateral_ref) + 1e-8

    theta_deg = (np.degrees(
        np.arctan2(radial @ lateral_ref, radial @ anterior_ref)
    ) + 360) % 360

    seg = np.zeros(len(verts), dtype=np.int32)

    apex_mask   = proj <= 0.10
    apical_mask = (proj > 0.10) & (proj <= 0.33)
    mid_mask    = (proj > 0.33) & (proj <= 0.67)
    basal_mask  = proj > 0.67

    seg[apex_mask]   = 17
    seg[apical_mask] = 13 + (theta_deg[apical_mask] // 90).astype(int) % 4
    seg[mid_mask]    =  7 + (theta_deg[mid_mask]    // 60).astype(int) % 6
    seg[basal_mask]  =  1 + (theta_deg[basal_mask]  // 60).astype(int) % 6

    return seg


# ─── Wall-thickness computation ──────────────────────────────
def compute_wall_thickness(endo_verts, epi_verts):
    """Per-vertex WT (mm) via epi KD-tree nearest-neighbour."""
    tree = cKDTree(epi_verts)
    dists, _ = tree.query(endo_verts)
    return dists


def aha_wall_thickness_map(endo_verts, epi_verts, faces):
    """
    Compute per-AHA-segment mean wall thickness.

    Parameters
    ----------
    endo_verts : (N, 3)
    epi_verts  : (N, 3)  — same topology as endo
    faces      : (F, 3)

    Returns
    -------
    wt_map  : dict {seg_id (1-17): mean_WT_mm}
    wt_per_v: (N,) per-vertex wall thickness
    seg_labels: (N,) AHA segment per vertex
    """
    apex_idx, _, base_center = find_apex_base(endo_verts, faces)
    wt_per_v   = compute_wall_thickness(endo_verts, epi_verts)
    seg_labels = assign_aha_segments(endo_verts, apex_idx, base_center)

    wt_map = {}
    for s in range(1, 18):
        mask = seg_labels == s
        wt_map[s] = float(wt_per_v[mask].mean()) if mask.any() else float('nan')

    return wt_map, wt_per_v, seg_labels


print('✅ AHA wall thickness functions ready')


## 15 · Bull's Eye Plot

In [ ]:
# AHA segment layout (angle = degrees CCW from top / anterior,
# as viewed from apex toward base)
_SEG_ANGLE = {
    1: 90, 2: 30, 3: -30, 4: -90, 5: -150, 6: 150,   # basal
    7: 90, 8: 30, 9: -30,10: -90,11: -150,12: 150,    # mid
   13: 90,14:  0,15: -90,16: 180,                      # apical
   17:  0,                                              # apex
}
_SEG_WIDTH = {**{s: 60 for s in range(1,13)},
              **{s: 90 for s in range(13,17)},
              17: 360}
_RING_R    = {**{s: (0.67, 1.00) for s in range(1, 7)},
              **{s: (0.34, 0.67) for s in range(7,13)},
              **{s: (0.10, 0.34) for s in range(13,17)},
              17: (0.00, 0.10)}
_RING_NAME = {**{s: 'Basal'  for s in range(1, 7)},
              **{s: 'Mid'    for s in range(7,13)},
              **{s: 'Apical' for s in range(13,17)},
              17: 'Apex'}


def _draw_single_bullseye(ax, wt_dict, title,
                           vmin=3.0, vmax=14.0, cmap='RdYlGn',
                           show_values=True):
    cmap_fn = plt.get_cmap(cmap)
    norm    = mcolors.Normalize(vmin=vmin, vmax=vmax)

    for seg_id in range(1, 18):
        val      = wt_dict.get(seg_id, float('nan'))
        color    = cmap_fn(norm(val)) if not np.isnan(val) else (.85,.85,.85,1)
        ang_c    = _SEG_ANGLE[seg_id]
        ang_w    = _SEG_WIDTH[seg_id]
        r_in, r_out = _RING_R[seg_id]

        # Wedge: matplotlib uses CCW from East; we define CW from top
        t1 = ang_c - ang_w / 2 + 90
        t2 = ang_c + ang_w / 2 + 90
        w  = mpatches.Wedge((0,0), r_out, t1, t2,
                             width=r_out - r_in,
                             facecolor=color, edgecolor='white', linewidth=0.6)
        ax.add_patch(w)

        if show_values and not np.isnan(val):
            ang_rad = np.radians(ang_c + 90)
            r_m     = (r_in + r_out) / 2
            x, y    = r_m * np.cos(ang_rad), r_m * np.sin(ang_rad)
            ax.text(x, y, f'{seg_id}\n{val:.1f}',
                    ha='center', va='center', fontsize=6.5,
                    fontweight='bold',
                    color='black' if val > (vmin + vmax) / 2 else 'white')

    ax.set_xlim(-1.15, 1.15); ax.set_ylim(-1.15, 1.15)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(title, fontsize=9, pad=6)
    sm = mcm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.02,
                 label='WT (mm)', shrink=0.85)


def bullseye_plot(wt_es, patient_id='Patient', es_range=(4, 16)):
    """
    Draw AHA 17-segment bull's eye for ES wall thickness.

    Parameters
    ----------
    wt_es : dict {seg_id 1-17: mean_WT_mm}
    """
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    _draw_single_bullseye(ax, wt_es, f'{patient_id} — ES WT (mm)',
                          vmin=es_range[0], vmax=es_range[1])
    plt.tight_layout()
    return fig


# ── Demo with synthetic ES data ──────────────────────────────
_demo_es = {s: np.random.uniform(5, 14) for s in range(1, 18)}
fig = bullseye_plot(_demo_es, patient_id='Demo patient (ES)')
plt.show()
print('✅ Bull\u2019s eye plot function ready (demo shown above)')


## 16 · Compute AHA WT from Cached ES Meshes

After retraining the INR, run inference on the ES frame
and save the output mesh as a `.npz` file in the same format as
the cache (`endo_verts`, `epi_verts`, `faces`).

This cell shows how to load any such file and compute the full AHA table.


In [ ]:
def compute_aha_from_npz(npz_path):
    """
    Load an endo/epi mesh .npz and return the AHA WT map.

    The .npz must contain: endo_verts, epi_verts, faces
    (all produced by process_patient_es or by INR inference).
    """
    d = np.load(npz_path)
    endo  = d['endo_verts'].astype(np.float64)
    epi   = d['epi_verts'].astype(np.float64)
    faces = d['faces'].astype(np.int32)

    wt_map, wt_per_v, seg_labels = aha_wall_thickness_map(endo, epi, faces)

    print(f'\n{Path(npz_path).name}  (phase={d.get("phase", "?")})')
    print('─' * 40)
    rings = [('Basal',  range(1, 7)),
             ('Mid',    range(7, 13)),
             ('Apical', range(13, 17)),
             ('Apex',   [17])]
    for ring_name, segs in rings:
        vals = [wt_map.get(s, float('nan')) for s in segs]
        mean_wt = np.nanmean(vals)
        seg_str = '  '.join(f'S{s}:{wt_map.get(s, float("nan")):.1f}' for s in segs)
        print(f'{ring_name:8s}  mean={mean_wt:.1f} mm  |  {seg_str}')

    return wt_map, wt_per_v, seg_labels


# ── Run on all ES cache files ─────────────────────────────────
es_files = sorted(Path(CFG['es_cache_dir']).glob('sample_*.npz'))
print(f'Found {len(es_files)} ES cache files')

all_wt_maps = []
for fp in tqdm(es_files[:10], desc='Computing AHA WT'):   # limit to 10 for demo
    try:
        wt_map, _, _ = compute_aha_from_npz(fp)
        all_wt_maps.append(wt_map)
    except Exception as e:
        print(f'  ✗ {fp.name}: {e}')

if all_wt_maps:
    print('\n── Population summary (mean ± std across patients) ──')
    for s in range(1, 18):
        vals = [m[s] for m in all_wt_maps if s in m and not np.isnan(m[s])]
        if vals:
            print(f'  S{s:2d}: {np.mean(vals):.2f} ± {np.std(vals):.2f} mm')


## 17 · Bull's Eye from ES Cache

**How to use after INR retraining:**

```python
# After running inference on an ES frame:
wt_es, _, _ = compute_aha_from_npz('patient001_ES.npz')
fig = bullseye_plot(wt_es, patient_id='Patient_001')
fig.savefig('bullseye_Patient_001.pdf', bbox_inches='tight')
```

The cell below demonstrates with cached ES samples.


In [ ]:
es_files = sorted(Path(CFG['es_cache_dir']).glob('sample_*.npz'))

if es_files:
    # Show bull's eye for first 3 ES samples
    n_show = min(3, len(es_files))
    for fp in es_files[:n_show]:
        wt_es, _, _ = compute_aha_from_npz(fp)
        fig = bullseye_plot(wt_es, patient_id=fp.stem)
        plt.show()
    print(f'\n✅ Showed {n_show} ES bull\'s eye plots')
else:
    print('No ES cache files found — run cell 11 first.')


## 18 · Batch Export — WT Table + Figures

In [ ]:
import csv

out_csv = Path(CFG['es_cache_dir']) / 'aha_wt_summary.csv'
out_fig_dir = Path(CFG['es_cache_dir']) / 'bullseye_figs'
out_fig_dir.mkdir(exist_ok=True)

fieldnames = ['file', 'phase'] + [f'S{s}' for s in range(1, 18)]

with open(out_csv, 'w', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    for fp in tqdm(sorted(Path(CFG['es_cache_dir']).glob('sample_*.npz')),
                   desc='Exporting WT table'):
        try:
            d = np.load(fp)
            endo  = d['endo_verts'].astype(np.float64)
            epi   = d['epi_verts'].astype(np.float64)
            faces = d['faces'].astype(np.int32)
            wt_map, _, _ = aha_wall_thickness_map(endo, epi, faces)
            row = {'file': fp.name, 'phase': str(d.get('phase', 'ES'))}
            row.update({f'S{s}': f'{wt_map.get(s, float("nan")):.3f}'
                        for s in range(1, 18)})
            writer.writerow(row)

            # Bull's eye figure
            fig = bullseye_plot(wt_map, patient_id=fp.stem)
            fig.savefig(out_fig_dir / f'{fp.stem}_bullseye.png',
                        dpi=120, bbox_inches='tight')
            plt.close(fig)
        except Exception as e:
            print(f'  ✗ {fp.name}: {e}')

print(f'✅ WT table saved: {out_csv}')
print(f'✅ Bull\u2019s eye figures: {out_fig_dir}')


## 19 · Cache Size Summary

In [ ]:
for label, cache_dir in [('ES cache', CFG['es_cache_dir'])]:
    p = Path(cache_dir)
    files = sorted(p.glob('sample_*.npz'))
    if files:
        total_bytes = sum(f.stat().st_size for f in files)
        print(f'{label}:')
        print(f'  Files    : {len(files)}')
        print(f'  Total    : {total_bytes / 1024**2:.1f} MB')
        print(f'  Avg/file : {total_bytes / len(files) / 1024:.1f} KB')
    else:
        print(f'{label}: empty or not found')


## 20 · Next Steps

> **ED data** (1,300 hearts) is already cached separately.
> This notebook only handles ES.

### 1. Get more ES training data
Only ACDC is available by default. To also use M&Ms and M&Ms-2:
```bash
kaggle datasets download -d <correct-slug-for-mnms>
```
Check the exact slug on Kaggle Datasets and update `CFG['mnms_dir']`.

### 2. Retrain the INR on ES cache
```python
# In training-model.ipynb
CFG['cache_dir'] = '/kaggle/working/es_occupancy_cache'
```
With ≥ 300 ES samples the model should no longer produce open apices on ES frames.

### 3. Run inference on ES frames
For each patient produce an output `.npz` file for the ES frame
containing `endo_verts`, `epi_verts`, `faces`.

### 4. Compute AHA wall thickness
```python
wt_es, _, _ = compute_aha_from_npz('patient001_ES.npz')
fig = bullseye_plot(wt_es, patient_id='Patient 001')
```

### 5. (Optional) Validate against manual measurements
Use the ACDC or M&Ms test set where manual contours are available.
Compare per-segment WT against manual measurements with MAE / Bland-Altman.

---
### Key SSM fitting parameters to tune (ES)

| Parameter | Value | Effect |
|---|---|---|
| `ssm_regularise_es` | **0.001** | Lower → more deformation allowed |
| `n_modes` | **100** | More modes → richer ES shape space |
| `ssm_max_iter` | **300** | More iters → better convergence |
| `epi_base_offset` | 10 mm | Larger → thicker myocardium at base |

| `epi_apex_offset` | 5 mm | Smaller → thinner at apex (realistic) |